# Polynomial Regression from scratch
Covers: polynomial features, "still linear in parameters", normal equation, gradient descent,
degree selection (under/overfitting), bias-variance, scaling, Ridge & Lasso, k-fold CV,
metrics (MAE, MSE, RMSE, R2, adj R2), poor extrapolation.

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(1)

## 1. Curved synthetic data (true function: cubic-ish + noise) and train/test split

In [ ]:
def true_f(x):
    return 0.5 * x ** 3 - 2 * x ** 2 - 3 * x + 10

n = 60
x_all = rng.uniform(-3, 4, n)
y_all = true_f(x_all) + rng.normal(0, 4, n)
idx = rng.permutation(n)
tr, te = idx[:40], idx[40:]
x_tr, y_tr, x_te, y_te = x_all[tr], y_all[tr], x_all[te], y_all[te]

## 2. Polynomial feature transform
x -> [x, x^2, ..., x^d]. (The bias column is added by the model.) Example from doc: x=2, d=3 -> [2,4,8].

In [ ]:
def poly_features(x, degree):
    """Return an (m x degree) matrix [x, x^2, ..., x^degree]."""
    x = np.asarray(x).reshape(-1)
    return np.column_stack([x ** p for p in range(1, degree + 1)])

print("x=2, degree=3 ->", poly_features([2], 3)[0])

def standardize(F, mu=None, sd=None):
    if mu is None:
        mu, sd = F.mean(0), F.std(0)
        sd[sd == 0] = 1
    return (F - mu) / sd, mu, sd

## 3. Metrics

In [ ]:
def mae(y, p): return np.mean(np.abs(y - p))
def mse(y, p): return np.mean((y - p) ** 2)
def rmse(y, p): return np.sqrt(mse(y, p))
def r2(y, p): return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)
def adj_r2(y, p, k): return 1 - (1 - r2(y, p)) * (len(y) - 1) / (len(y) - k - 1)

## 4. Training: Normal equation (with optional Ridge), gradient descent, Lasso (coordinate descent)
Same algorithms as linear regression - only the features changed.

In [ ]:
def fit_normal(F, y, lam=0.0):
    """beta = (X^T X + lam I)^-1 X^T y; intercept unpenalised. Uses lstsq-like solve w/ tiny jitter."""
    Xb = np.c_[np.ones(len(F)), F]
    P = lam * np.eye(Xb.shape[1]); P[0, 0] = 0
    A = Xb.T @ Xb + P + 1e-10 * np.eye(Xb.shape[1])
    return np.linalg.solve(A, Xb.T @ y)

def fit_gd(F, y, alpha=0.05, n_iters=3000):
    """Batch GD on J = 1/(2m) sum (pred - y)^2. Returns beta (bias first) and cost history."""
    m = len(y)
    Xb = np.c_[np.ones(m), F]
    beta = np.zeros(Xb.shape[1]); hist = []
    for _ in range(n_iters):
        err = Xb @ beta - y
        beta -= alpha * Xb.T @ err / m
        hist.append(np.sum(err ** 2) / (2 * m))
    return beta, np.array(hist)

def soft_threshold(z, t):
    return np.sign(z) * np.maximum(np.abs(z) - t, 0.0)

def fit_lasso(F, y, lam=0.1, n_iters=300):
    """Coordinate descent for (1/2m)||y - b0 - F b||^2 + lam*||b||_1 (F must be standardised)."""
    m, k = F.shape
    b = np.zeros(k); b0 = y.mean()
    for _ in range(n_iters):
        for j in range(k):
            r = y - b0 - F @ b + F[:, j] * b[j]
            b[j] = soft_threshold(F[:, j] @ r / m, lam) / (F[:, j] @ F[:, j] / m)
    return np.r_[b0, b]

def predict(F, beta):
    return np.c_[np.ones(len(F)), F] @ beta

## 5. Linear vs polynomial; normal equation vs gradient descent

In [ ]:
xs = np.linspace(-3, 4, 200)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for d, c in [(1, 'tab:blue'), (3, 'tab:green'), (12, 'tab:red')]:
    F_tr, mu, sd = standardize(poly_features(x_tr, d))
    beta = fit_normal(F_tr, y_tr)
    ax[0].plot(xs, predict((poly_features(xs, d) - mu) / sd, beta), c=c, label=f'degree {d}')
    print(f"degree {d:2d}: train RMSE={rmse(y_tr, predict(F_tr, beta)):.2f}  "
          f"test RMSE={rmse(y_te, predict((poly_features(x_te, d) - mu) / sd, beta)):.2f}")
ax[0].scatter(x_tr, y_tr, c='k', s=12); ax[0].scatter(x_te, y_te, c='orange', s=18)
ax[0].plot(xs, true_f(xs), 'k--', lw=1, label='truth'); ax[0].set_ylim(-40, 60)
ax[0].legend(); ax[0].set_title('Underfit / good / overfit')

d = 3
F_tr, mu, sd = standardize(poly_features(x_tr, d))
b_ne = fit_normal(F_tr, y_tr)
b_gd, hist = fit_gd(F_tr, y_tr)
print("normal eq beta:", b_ne.round(3)); print("GD beta:       ", b_gd.round(3))
ax[1].plot(hist); ax[1].set_yscale('log'); ax[1].set_title('GD cost (degree 3, scaled)')
ax[1].set_xlabel('iteration')

# without scaling GD struggles (x^3 has huge range)
F_raw = poly_features(x_tr, d)
b_raw, h_raw = fit_gd(F_raw, y_tr, alpha=0.001, n_iters=3000)
ax[2].plot(hist, label='scaled'); ax[2].plot(h_raw, label='unscaled'); ax[2].set_yscale('log')
ax[2].legend(); ax[2].set_title('Scaling matters for GD')
plt.tight_layout(); plt.savefig('plots/01_fits_and_gd.png', dpi=120); plt.close()

## 6. Choosing the degree: train vs test error (bias-variance) and k-fold CV

In [ ]:
def kfold_cv(x, y, degree, k=5, lam=0.0, seed=0):
    idx = np.random.default_rng(seed).permutation(len(x)); folds = np.array_split(idx, k)
    errs = []
    for i in range(k):
        v = folds[i]; t = np.concatenate([folds[j] for j in range(k) if j != i])
        F, mu, sd = standardize(poly_features(x[t], degree))
        beta = fit_normal(F, y[t], lam)
        errs.append(rmse(y[v], predict((poly_features(x[v], degree) - mu) / sd, beta)))
    return np.mean(errs)

degrees = range(1, 13)
tr_err, te_err, cv_err = [], [], []
for d in degrees:
    F, mu, sd = standardize(poly_features(x_tr, d)); beta = fit_normal(F, y_tr)
    tr_err.append(rmse(y_tr, predict(F, beta)))
    te_err.append(rmse(y_te, predict((poly_features(x_te, d) - mu) / sd, beta)))
    cv_err.append(kfold_cv(x_tr, y_tr, d))
best_d = list(degrees)[int(np.argmin(cv_err))]
print("CV-selected degree:", best_d)
plt.figure(figsize=(6, 4))
plt.plot(degrees, tr_err, 'o-', label='train'); plt.plot(degrees, te_err, 's-', label='test')
plt.plot(degrees, cv_err, '^-', label='5-fold CV'); plt.yscale('log'); plt.xlabel('degree'); plt.ylabel('RMSE')
plt.legend(); plt.title('Bias-variance vs degree')
plt.tight_layout(); plt.savefig('plots/02_degree_selection.png', dpi=120); plt.close()

## 7. Regularisation for a high degree (12): Ridge and Lasso

In [ ]:
d = 12
F_tr, mu, sd = standardize(poly_features(x_tr, d)); F_te = (poly_features(x_te, d) - mu) / sd
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(x_tr, y_tr, c='k', s=12)
Fx = (poly_features(xs, d) - mu) / sd
for name, beta in [('OLS', fit_normal(F_tr, y_tr)), ('Ridge lam=1', fit_normal(F_tr, y_tr, 1.0)),
                   ('Lasso lam=0.5', fit_lasso(F_tr, y_tr, 0.5))]:
    ax[0].plot(xs, predict(Fx, beta), label=name)
    print(f"{name:14s} test RMSE={rmse(y_te, predict(F_te, beta)):8.2f}  nonzero coefs={(np.abs(beta[1:]) > 1e-8).sum()}")
ax[0].set_ylim(-40, 60); ax[0].legend(); ax[0].set_title('Degree 12 with regularisation')
lams = np.logspace(-3, 1, 15)
paths = np.array([fit_lasso(F_tr, y_tr, l)[1:] for l in lams])
for j in range(d): ax[1].semilogx(lams, paths[:, j])
ax[1].set_xlabel('lambda'); ax[1].set_title('Lasso coefficient paths'); 
plt.tight_layout(); plt.savefig('plots/03_regularisation.png', dpi=120); plt.close()

## 8. Evaluation report and extrapolation
Polynomials extrapolate badly outside the training range.

In [ ]:
F3, mu3, sd3 = standardize(poly_features(x_tr, 3)); b3 = fit_normal(F3, y_tr)
p = predict((poly_features(x_te, 3) - mu3) / sd3, b3)
print(f"Degree 3 test: MAE={mae(y_te, p):.2f} MSE={mse(y_te, p):.2f} RMSE={rmse(y_te, p):.2f} "
      f"R2={r2(y_te, p):.3f} adjR2={adj_r2(y_te, p, 3):.3f}")
xe = np.linspace(-6, 7, 200)
plt.figure(figsize=(6, 4))
for dd in (3, 9):
    F, m_, s_ = standardize(poly_features(x_tr, dd)); b = fit_normal(F, y_tr)
    plt.plot(xe, predict((poly_features(xe, dd) - m_) / s_, b), label=f'degree {dd}')
plt.plot(xe, true_f(xe), 'k--', label='truth'); plt.scatter(x_tr, y_tr, s=10, c='k')
plt.ylim(-150, 150); plt.axvspan(-3, 4, alpha=0.08); plt.legend(); plt.title('Extrapolation fails')
plt.tight_layout(); plt.savefig('plots/04_extrapolation.png', dpi=120); plt.close()

## 9. Sanity check against np.polyfit (degree 3)

In [ ]:
coef = np.polyfit(x_tr, y_tr, 3)
print("polyfit pred diff:", np.max(np.abs(np.polyval(coef, x_te) - p)).round(6))